# Chapter 19 — Generating Text From N-Gram Models

Chapter 18 trained general n-gram models with configurable context length.

This chapter focuses on the generation process that repeatedly selects and appends one token.

By the end of this chapter, you should be able to:

- explain the one-token generation loop;
- maintain a rolling context window;
- sample reproducibly from context probabilities;
- trace the probability source at every step;
- begin with a prompt shorter than the maximum context;
- back off through shorter observed contexts;
- use unigram or uniform fallback when no context is available;
- compare sampled and greedy decoding; and
- distinguish fitting a model from generating with it.

This chapter uses the existing count-based n-gram family rather than introducing a new model family.


## The Autoregressive Generation Loop

An n-gram generator repeats an ordered loop:

1. Select the most recent usable context.

2. Retrieve next-token probabilities.

3. Choose one next token.

4. Append that token to the generated sequence.

5. Slide the context window forward and repeat.

Generating one token and then conditioning on the updated sequence is called **autoregressive generation**.


## A Rolling Context Window

With context length two, starting from `"th"` and appending `"e"` changes the context from `"th"` to `"he"`.

Appending a space then changes the context from `"he"` to `"e "`.

Older tokens fall out of the fixed-width window as new tokens enter it.

The complete generated text can keep growing even though the model consults only its final context window.


## Train on a Repeated Local Corpus

The Chapter 18 fixture contains repeated phrases that make context lookup and generated fragments easy to inspect.


In [1]:
corpus_lines = [
    "the cat sat on the mat.",
    "the cat sat on the rug.",
    "the cat slept on the mat.",
    "the dog sat on the mat.",
    "the dog ran in the yard.",
    "the dog slept on the rug.",
    "the bird sat in the tree.",
    "the bird sang in the tree.",
    "the child sat on the rug.",
    "the child ran in the yard.",
    "the cat ran in the yard.",
    "the dog ate the food.",
    "the cat ate the food.",
    "the bird ate the seed.",
    "the child ate the food.",
    "the cat looked at the dog.",
    "the small cat sat on the mat.",
    "the small dog ran in the yard.",
    "the small bird sang in the tree.",
    "the happy child sat on the rug.",
    "the happy dog slept on the mat.",
    "the dog looked at the cat.",
    "the bird looked at the child.",
    "the child looked at the bird.",
]

training_text = "\n".join(corpus_lines)

print(training_text)
print()
print("Training lines:", len(corpus_lines))
print("Training characters:", len(training_text))

the cat sat on the mat.
the cat sat on the rug.
the cat slept on the mat.
the dog sat on the mat.
the dog ran in the yard.
the dog slept on the rug.
the bird sat in the tree.
the bird sang in the tree.
the child sat on the rug.
the child ran in the yard.
the cat ran in the yard.
the dog ate the food.
the cat ate the food.
the bird ate the seed.
the child ate the food.
the cat looked at the dog.
the small cat sat on the mat.
the small dog ran in the yard.
the small bird sang in the tree.
the happy child sat on the rug.
the happy dog slept on the mat.
the dog looked at the cat.
the bird looked at the child.
the child looked at the bird.

Training lines: 24
Training characters: 642


## Build the Character Tokenizer

The vocabulary limits both prompts and generated output to characters observed during training.


In [2]:
def build_character_mappings(
    text: str,
) -> tuple[list[str], dict[str, int], dict[int, str]]:
    vocabulary = sorted(set(text))
    character_to_id = {
        character: character_id for character_id, character in enumerate(vocabulary)
    }
    id_to_character = {
        character_id: character for character, character_id in character_to_id.items()
    }
    return vocabulary, character_to_id, id_to_character


character_vocabulary, character_to_id, id_to_character = build_character_mappings(
    training_text
)

print("Vocabulary size:", len(character_vocabulary))
print([repr(character) for character in character_vocabulary])

Vocabulary size: 23
["'\\n'", "' '", "'.'", "'a'", "'b'", "'c'", "'d'", "'e'", "'f'", "'g'", "'h'", "'i'", "'k'", "'l'", "'m'", "'n'", "'o'", "'p'", "'r'", "'s'", "'t'", "'u'", "'y'"]


## Encode and Decode IDs

The exact training round trip checks the tokenizer before any model rows are fitted.


In [3]:
def encode_characters(
    text: str,
    character_to_id: dict[str, int],
) -> list[int]:
    token_ids = []
    for position, character in enumerate(text):
        if character not in character_to_id:
            raise ValueError(f"Unknown character {character!r} at position {position}.")
        token_ids.append(character_to_id[character])
    return token_ids


def decode_character_ids(
    token_ids: list[int],
    id_to_character: dict[int, str],
) -> str:
    characters = []
    for position, token_id in enumerate(token_ids):
        if token_id not in id_to_character:
            raise ValueError(f"Unknown token ID {token_id} at position {position}.")
        characters.append(id_to_character[token_id])
    return "".join(characters)


training_token_ids = encode_characters(training_text, character_to_id)

assert decode_character_ids(training_token_ids, id_to_character) == training_text
print("Training token count:", len(training_token_ids))
print("First 80 token IDs:", training_token_ids[:80])

Training token count: 642
First 80 token IDs: [20, 10, 7, 1, 5, 3, 20, 1, 19, 3, 20, 1, 16, 15, 1, 20, 10, 7, 1, 14, 3, 20, 2, 0, 20, 10, 7, 1, 5, 3, 20, 1, 19, 3, 20, 1, 16, 15, 1, 20, 10, 7, 1, 18, 21, 9, 2, 0, 20, 10, 7, 1, 5, 3, 20, 1, 19, 13, 7, 17, 20, 1, 16, 15, 1, 20, 10, 7, 1, 14, 3, 20, 2, 0, 20, 10, 7, 1, 6, 16]


## Count Targets After General Contexts

Each context tuple maps to counts of the target token that immediately followed it.


In [4]:
from collections import Counter  # noqa: I001


Context = tuple[int, ...]


def build_context_counts(
    token_ids: list[int],
    context_length: int,
) -> dict[Context, Counter[int]]:
    if context_length < 1:
        raise ValueError("context_length must be at least 1.")
    if context_length >= len(token_ids):
        raise ValueError("context_length must be smaller than the token sequence.")

    context_to_target_counts: dict[Context, Counter[int]] = {}
    for start_position in range(len(token_ids) - context_length):
        context = tuple(token_ids[start_position : start_position + context_length])
        target_id = token_ids[start_position + context_length]
        if context not in context_to_target_counts:
            context_to_target_counts[context] = Counter()
        context_to_target_counts[context][target_id] += 1
    return context_to_target_counts

## Smooth Observed Context Rows

Add-alpha smoothing gives every known vocabulary target positive probability after an observed context.

It does not create a row for a context tuple that never appeared.


In [5]:
def assert_valid_probability_distribution(probabilities: list[float]) -> None:
    if not probabilities:
        raise ValueError("A probability distribution cannot be empty.")
    if any(probability < 0 for probability in probabilities):
        raise ValueError("Probabilities cannot be negative.")
    if abs(sum(probabilities) - 1.0) > 1e-12:
        raise ValueError("Probabilities must sum to 1.")


def smooth_target_counts(
    target_counts: Counter[int],
    vocabulary_size: int,
    alpha: float,
) -> list[float]:
    if alpha <= 0:
        raise ValueError("alpha must be greater than 0.")
    denominator = sum(target_counts.values()) + alpha * vocabulary_size
    probabilities = [
        (target_counts[target_id] + alpha) / denominator
        for target_id in range(vocabulary_size)
    ]
    assert_valid_probability_distribution(probabilities)
    return probabilities


def build_context_probabilities(
    context_counts: dict[Context, Counter[int]],
    vocabulary_size: int,
    alpha: float,
) -> dict[Context, list[float]]:
    return {
        context: smooth_target_counts(target_counts, vocabulary_size, alpha)
        for context, target_counts in context_counts.items()
    }

## Build Unigram and Uniform Fallbacks

Unigram fallback preserves overall character frequency, while uniform fallback assigns equal probability to every vocabulary token.


In [6]:
def count_token_ids(token_ids: list[int], vocabulary_size: int) -> list[int]:
    counts = [0 for _ in range(vocabulary_size)]
    for token_id in token_ids:
        if not 0 <= token_id < vocabulary_size:
            raise ValueError(f"Invalid token ID {token_id}.")
        counts[token_id] += 1
    return counts


def smooth_count_list(counts: list[int], alpha: float) -> list[float]:
    if not counts:
        raise ValueError("counts cannot be empty.")
    if alpha <= 0:
        raise ValueError("alpha must be greater than 0.")
    denominator = sum(counts) + alpha * len(counts)
    probabilities = [(count + alpha) / denominator for count in counts]
    assert_valid_probability_distribution(probabilities)
    return probabilities


def uniform_probabilities(vocabulary_size: int) -> list[float]:
    if vocabulary_size < 1:
        raise ValueError("vocabulary_size must be at least 1.")
    probabilities = [1.0 / vocabulary_size for _ in range(vocabulary_size)]
    assert_valid_probability_distribution(probabilities)
    return probabilities


VOCABULARY_SIZE = len(character_vocabulary)
ALPHA = 0.1
unigram_probabilities = smooth_count_list(
    count_token_ids(training_token_ids, VOCABULARY_SIZE),
    ALPHA,
)
uniform_fallback_probabilities = uniform_probabilities(VOCABULARY_SIZE)

## Fit Every Backoff Length

To support shorter-context backoff from a maximum length of eight, the model needs fitted dictionaries for lengths one through eight.


In [7]:
def build_probabilities_by_length(
    token_ids: list[int],
    maximum_context_length: int,
    vocabulary_size: int,
    alpha: float,
) -> dict[int, dict[Context, list[float]]]:
    if maximum_context_length < 1:
        raise ValueError("maximum_context_length must be at least 1.")

    return {
        context_length: build_context_probabilities(
            build_context_counts(token_ids, context_length),
            vocabulary_size,
            alpha,
        )
        for context_length in range(1, maximum_context_length + 1)
    }


MAXIMUM_CONTEXT_LENGTH = 8
context_probabilities_by_length = build_probabilities_by_length(
    training_token_ids,
    MAXIMUM_CONTEXT_LENGTH,
    VOCABULARY_SIZE,
    ALPHA,
)

for context_length, context_rows in context_probabilities_by_length.items():
    print(f"length {context_length}: {len(context_rows)} observed contexts")

length 1: 23 observed contexts
length 2: 74 observed contexts
length 3: 117 observed contexts
length 4: 153 observed contexts
length 5: 186 observed contexts
length 6: 221 observed contexts
length 7: 254 observed contexts
length 8: 292 observed contexts


## Define the Backoff Model

The model searches from the longest currently available suffix down to length one.

If none of those suffixes has an observed row, it uses the selected context-free fallback.

The returned source string makes every decision inspectable.


In [8]:
class BackoffNGramCharacterModel:
    def __init__(
        self,
        maximum_context_length: int,
        context_probabilities_by_length: dict[
            int,
            dict[Context, list[float]],
        ],
        unigram_probabilities: list[float],
        uniform_probabilities: list[float],
        fallback_strategy: str,
    ):
        if maximum_context_length < 1:
            raise ValueError("maximum_context_length must be at least 1.")
        if fallback_strategy not in {"unigram", "uniform"}:
            raise ValueError("fallback_strategy must be 'unigram' or 'uniform'.")

        self.maximum_context_length = maximum_context_length
        self.context_probabilities_by_length = context_probabilities_by_length
        self.unigram_probabilities = unigram_probabilities
        self.uniform_probabilities = uniform_probabilities
        self.fallback_strategy = fallback_strategy

        assert_valid_probability_distribution(unigram_probabilities)
        assert_valid_probability_distribution(uniform_probabilities)
        for rows in context_probabilities_by_length.values():
            for probabilities in rows.values():
                assert_valid_probability_distribution(probabilities)

    def fallback_probabilities(self) -> list[float]:
        if self.fallback_strategy == "unigram":
            return self.unigram_probabilities
        return self.uniform_probabilities

    def probabilities_and_source(
        self,
        input_token_ids: list[int],
    ) -> tuple[list[float], str, Context]:
        maximum_usable_length = min(
            self.maximum_context_length,
            len(input_token_ids),
        )
        for context_length in range(maximum_usable_length, 0, -1):
            context = tuple(input_token_ids[-context_length:])
            rows = self.context_probabilities_by_length.get(context_length, {})
            if context in rows:
                return rows[context], f"observed length {context_length}", context
        return self.fallback_probabilities(), f"{self.fallback_strategy} fallback", ()


generation_model = BackoffNGramCharacterModel(
    MAXIMUM_CONTEXT_LENGTH,
    context_probabilities_by_length,
    unigram_probabilities,
    uniform_fallback_probabilities,
    fallback_strategy="unigram",
)

## Inspect a Single Probability Lookup

Before generation, inspect which suffix row is selected for a familiar prompt.


In [9]:
prompt = "the cat "
prompt_ids = encode_characters(prompt, character_to_id)
probabilities, source, context = generation_model.probabilities_and_source(prompt_ids)

print("Prompt:", repr(prompt))
print("Source:", source)
print("Context used:", repr(decode_character_ids(list(context), id_to_character)))
print("Probability sum:", sum(probabilities))

Prompt: 'the cat '
Source: observed length 8
Context used: 'the cat '
Probability sum: 1.0


## Sample One Token Reproducibly

Sampling follows the distribution rather than always selecting its largest entry.


In [10]:
import random  # noqa: I001


RANDOM_SEED = 19


def sample_token_id(
    probabilities: list[float],
    random_generator: random.Random,
) -> int:
    assert_valid_probability_distribution(probabilities)
    return random_generator.choices(
        range(len(probabilities)),
        weights=probabilities,
        k=1,
    )[0]


random_generator = random.Random(RANDOM_SEED)
sampled_id = sample_token_id(probabilities, random_generator)

print("Sampled token ID:", sampled_id)
print("Sampled character:", repr(id_to_character[sampled_id]))
print("Sampled probability:", probabilities[sampled_id])

Sampled token ID: 19
Sampled character: 's'
Sampled probability: 0.37349397590361444


## Generate While Maintaining Token IDs

The generator encodes the prompt once and then appends sampled IDs directly.

This makes the rolling suffix explicit and avoids re-encoding the entire growing string at every step.


In [11]:
def generate_text(
    starting_text: str,
    number_of_new_tokens: int,
    model: BackoffNGramCharacterModel,
    character_to_id: dict[str, int],
    id_to_character: dict[int, str],
    random_seed: int,
) -> tuple[str, Counter[str]]:
    if number_of_new_tokens < 0:
        raise ValueError("number_of_new_tokens cannot be negative.")

    generated_ids = encode_characters(starting_text, character_to_id)
    random_generator = random.Random(random_seed)
    source_counts: Counter[str] = Counter()

    for _ in range(number_of_new_tokens):
        probabilities, source, context = model.probabilities_and_source(generated_ids)
        source_counts[source] += 1
        generated_ids.append(sample_token_id(probabilities, random_generator))

    return decode_character_ids(generated_ids, id_to_character), source_counts


generated_text, source_counts = generate_text(
    starting_text="the ",
    number_of_new_tokens=160,
    model=generation_model,
    character_to_id=character_to_id,
    id_to_character=id_to_character,
    random_seed=RANDOM_SEED,
)

print(generated_text)
print()
print(repr(generated_text))
print("Probability sources:", dict(source_counts))

the me hayag imaild bcat sleat.
thn.
the hachfo
the catlon the matat s.
the ch mad at tn in 
thhildsat on on n the yard.
tat optog n thepchilard.
theugug.
thn hg.
t

'the me hayag imaild bcat sleat.\nthn.\nthe hachfo\nthe catlon the matat s.\nthe ch mad at tn in \nthhildsat on on n the yard.\ntat optog n thepchilard.\ntheugug.\nthn hg.\nt'
Probability sources: {'observed length 4': 21, 'observed length 5': 14, 'observed length 1': 32, 'observed length 2': 36, 'observed length 3': 25, 'observed length 6': 11, 'observed length 7': 8, 'observed length 8': 13}


The prompt remains at the beginning, and each sampled ID becomes part of the next rolling context.


## Trace Every Generation Step

The trace prints the context actually used, its probability source, the sampled character, and the new maximum-width suffix.


In [12]:
def trace_generation(
    starting_text: str,
    number_of_new_tokens: int,
    model: BackoffNGramCharacterModel,
    character_to_id: dict[str, int],
    id_to_character: dict[int, str],
    random_seed: int,
) -> str:
    generated_ids = encode_characters(starting_text, character_to_id)
    random_generator = random.Random(random_seed)

    for step in range(number_of_new_tokens):
        probabilities, source, context = model.probabilities_and_source(generated_ids)
        sampled_id = sample_token_id(probabilities, random_generator)
        generated_ids.append(sampled_id)
        new_suffix = generated_ids[-model.maximum_context_length :]
        print(f"step {step + 1:>2}")
        print(
            "  context used: ",
            repr(decode_character_ids(list(context), id_to_character)),
        )
        print("  source:       ", source)
        print("  sampled:      ", repr(id_to_character[sampled_id]))
        print("  probability:  ", probabilities[sampled_id])
        print(
            "  new suffix:   ", repr(decode_character_ids(new_suffix, id_to_character))
        )
        print()

    return decode_character_ids(generated_ids, id_to_character)


traced_text = trace_generation(
    starting_text="the ",
    number_of_new_tokens=12,
    model=generation_model,
    character_to_id=character_to_id,
    id_to_character=id_to_character,
    random_seed=RANDOM_SEED,
)
print("Final traced text:", repr(traced_text))

step  1
  context used:  'the '
  source:        observed length 4
  sampled:       'm'
  probability:   0.10139165009940358
  new suffix:    'the m'

step  2
  context used:  'the m'
  source:        observed length 5
  sampled:       'e'
  probability:   0.0136986301369863
  new suffix:    'the me'

step  3
  context used:  'e'
  source:        observed length 1
  sampled:       ' '
  probability:   0.7518037518037518
  new suffix:    'the me '

step  4
  context used:  'e '
  source:        observed length 2
  sampled:       'h'
  probability:   0.038674033149171276
  new suffix:    'the me h'

step  5
  context used:  'e h'
  source:        observed length 3
  sampled:       'a'
  probability:   0.48837209302325574
  new suffix:    'he me ha'

step  6
  context used:  'e ha'
  source:        observed length 4
  sampled:       'y'
  probability:   0.02325581395348837
  new suffix:    'e me hay'

step  7
  context used:  'y'
  source:        observed length 1
  sampled:       'a'
  p

## Isolate a Two-Token Rolling Window

A maximum context length of two makes each one-position slide especially easy to see.


In [13]:
probabilities_up_to_length_two = build_probabilities_by_length(
    training_token_ids,
    maximum_context_length=2,
    vocabulary_size=VOCABULARY_SIZE,
    alpha=ALPHA,
)
model_length_two = BackoffNGramCharacterModel(
    2,
    probabilities_up_to_length_two,
    unigram_probabilities,
    uniform_fallback_probabilities,
    fallback_strategy="unigram",
)

traced_length_two = trace_generation(
    starting_text="th",
    number_of_new_tokens=10,
    model=model_length_two,
    character_to_id=character_to_id,
    id_to_character=id_to_character,
    random_seed=RANDOM_SEED,
)
print("Final length-two text:", repr(traced_length_two))

step  1
  context used:  'th'
  source:        observed length 2
  sampled:       'e'
  probability:   0.9562624254473162
  new suffix:    'he'

step  2
  context used:  'he'
  source:        observed length 2
  sampled:       ' '
  probability:   0.9562624254473162
  new suffix:    'e '

step  3
  context used:  'e '
  source:        observed length 2
  sampled:       'h'
  probability:   0.038674033149171276
  new suffix:    ' h'

step  4
  context used:  ' h'
  source:        observed length 2
  sampled:       'a'
  probability:   0.48837209302325574
  new suffix:    'ha'

step  5
  context used:  'ha'
  source:        observed length 2
  sampled:       'o'
  probability:   0.02325581395348837
  new suffix:    'ao'

step  6
  context used:  'o'
  source:        observed length 1
  sampled:       'u'
  probability:   0.0030030030030030034
  new suffix:    'ou'

step  7
  context used:  'u'
  source:        observed length 1
  sampled:       'g'
  probability:   0.6507936507936507
  n

## Print Rolling Windows on Completed Text

The post-generation diagram confirms that every target follows the immediately preceding fixed-width slice.


In [14]:
def print_rolling_windows(
    text: str,
    context_length: int,
    max_windows: int,
) -> None:
    if context_length < 1:
        raise ValueError("context_length must be at least 1.")
    if max_windows < 0:
        raise ValueError("max_windows cannot be negative.")

    for start_position in range(min(max_windows, len(text) - context_length)):
        context_text = text[start_position : start_position + context_length]
        target = text[start_position + context_length]
        print(
            f"window {start_position:>2}: "
            f"{' ' * start_position}[{context_text}] → {target!r}"
        )


print("Text:", repr(traced_length_two))
print_rolling_windows(traced_length_two, context_length=2, max_windows=10)

Text: 'the haoug.\nt'
window  0: [th] → 'e'
window  1:  [he] → ' '
window  2:   [e ] → 'h'
window  3:    [ h] → 'a'
window  4:     [ha] → 'o'
window  5:      [ao] → 'u'
window  6:       [ou] → 'g'
window  7:        [ug] → '.'
window  8:         [g.] → '\n'
window  9:          [.
] → 't'


## Grow From a Prompt Shorter Than the Maximum

The search begins with the available one-token suffix and can use progressively longer contexts as generated IDs accumulate.


In [15]:
short_prompt_text = trace_generation(
    starting_text="t",
    number_of_new_tokens=10,
    model=generation_model,
    character_to_id=character_to_id,
    id_to_character=id_to_character,
    random_seed=RANDOM_SEED,
)
print("Generated from short prompt:", repr(short_prompt_text))

step  1
  context used:  't'
  source:        observed length 1
  sampled:       'h'
  probability:   0.5705812574139977
  new suffix:    'th'

step  2
  context used:  'th'
  source:        observed length 2
  sampled:       'e'
  probability:   0.9562624254473162
  new suffix:    'the'

step  3
  context used:  'the'
  source:        observed length 3
  sampled:       ' '
  probability:   0.9562624254473162
  new suffix:    'the '

step  4
  context used:  'the '
  source:        observed length 4
  sampled:       'f'
  probability:   0.061630218687872766
  new suffix:    'the f'

step  5
  context used:  'the f'
  source:        observed length 5
  sampled:       'o'
  probability:   0.5849056603773585
  new suffix:    'the fo'

step  6
  context used:  'the fo'
  source:        observed length 6
  sampled:       'y'
  probability:   0.018867924528301886
  new suffix:    'the foy'

step  7
  context used:  'y'
  source:        observed length 1
  sampled:       'a'
  probability:   

## Reject Unknown Prompt Characters

A fixed character vocabulary cannot encode a prompt containing an unseen character such as `"z"`.


In [16]:
try:
    generate_text(
        starting_text="the zebra",
        number_of_new_tokens=10,
        model=generation_model,
        character_to_id=character_to_id,
        id_to_character=id_to_character,
        random_seed=RANDOM_SEED,
    )
except ValueError as error:
    print("Caught expected error:", error)

Caught expected error: Unknown character 'z' at position 4.


Unknown-token handling belongs to tokenization and vocabulary design rather than to n-gram backoff.


## Force Shorter-Context Backoff

The unusual suffix below is made from known characters but lacks an observed length-eight row.

The model reports the longest shorter suffix it can use.


In [17]:
unusual_prompt = "the mat the"
unusual_ids = encode_characters(unusual_prompt, character_to_id)
unusual_probabilities, unusual_source, unusual_context = (
    generation_model.probabilities_and_source(unusual_ids)
)

print("Prompt:", repr(unusual_prompt))
print("Source:", unusual_source)
print(
    "Context used:", repr(decode_character_ids(list(unusual_context), id_to_character))
)
print("Probability sum:", sum(unusual_probabilities))

Prompt: 'the mat the'
Source: observed length 6
Context used: 'at the'
Probability sum: 0.9999999999999999


## Trace After the Unusual Prompt

Each sampled token can lead back into longer observed contexts or trigger further backoff.


In [18]:
unusual_generated_text = trace_generation(
    starting_text=unusual_prompt,
    number_of_new_tokens=8,
    model=generation_model,
    character_to_id=character_to_id,
    id_to_character=id_to_character,
    random_seed=RANDOM_SEED,
)
print("Generated text:", repr(unusual_generated_text))

step  1
  context used:  'at the'
  source:        observed length 6
  sampled:       '.'
  probability:   0.015873015873015872
  new suffix:    'mat the.'

step  2
  context used:  'e.'
  source:        observed length 2
  sampled:       'i'
  probability:   0.018867924528301886
  new suffix:    'at the.i'

step  3
  context used:  'i'
  source:        observed length 1
  sampled:       'n'
  probability:   0.3333333333333333
  new suffix:    't the.in'

step  4
  context used:  'in'
  source:        observed length 2
  sampled:       ' '
  probability:   0.7634408602150536
  new suffix:    ' the.in '

step  5
  context used:  'in '
  source:        observed length 3
  sampled:       't'
  probability:   0.7634408602150536
  new suffix:    'the.in t'

step  6
  context used:  'in t'
  source:        observed length 4
  sampled:       'y'
  probability:   0.01075268817204301
  new suffix:    'he.in ty'

step  7
  context used:  'y'
  source:        observed length 1
  sampled:       'a

## Force a True Context-Free Fallback

An empty prompt supplies no suffix at any positive context length, so the configured unigram or uniform fallback is used immediately.


In [19]:
empty_probabilities, empty_source, empty_context = (
    generation_model.probabilities_and_source([])
)

print("Source for empty prompt:", empty_source)
print("Context:", empty_context)
print("Probability sum:", sum(empty_probabilities))
assert empty_source == "unigram fallback"

Source for empty prompt: unigram fallback
Context: ()
Probability sum: 1.0


## Compare Unigram and Uniform Fallbacks Directly

The empty prompt makes the fallback choice observable because no shorter context can intervene.


In [20]:
uniform_fallback_model = BackoffNGramCharacterModel(
    MAXIMUM_CONTEXT_LENGTH,
    context_probabilities_by_length,
    unigram_probabilities,
    uniform_fallback_probabilities,
    fallback_strategy="uniform",
)

unigram_row, unigram_source, unigram_context = (
    generation_model.probabilities_and_source([])
)
uniform_row, uniform_source, uniform_context = (
    uniform_fallback_model.probabilities_and_source([])
)

print("Unigram source:", unigram_source)
print("Uniform source:", uniform_source)
print(
    "Most likely unigram character:",
    repr(id_to_character[max(range(VOCABULARY_SIZE), key=unigram_row.__getitem__)]),
)
print("Uniform entries all equal:", len(set(uniform_row)) == 1)
assert unigram_row != uniform_row

Unigram source: unigram fallback
Uniform source: uniform fallback
Most likely unigram character: ' '
Uniform entries all equal: True


Unigram fallback is more informed because it reflects overall training frequency, whereas uniform fallback ignores the corpus frequencies.


## Inspect Top Next-Token Options

Looking at ranked probabilities explains the choices available before randomness selects one token.


In [21]:
def print_top_next_characters(
    starting_text: str,
    model: BackoffNGramCharacterModel,
    character_to_id: dict[str, int],
    id_to_character: dict[int, str],
    top_k: int,
) -> None:
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")

    input_ids = encode_characters(starting_text, character_to_id)
    probabilities, source, context = model.probabilities_and_source(input_ids)
    ranked_ids = sorted(
        range(len(probabilities)),
        key=probabilities.__getitem__,
        reverse=True,
    )
    print("Starting text:", repr(starting_text))
    print("Source:", source)
    print("Context used:", repr(decode_character_ids(list(context), id_to_character)))
    for token_id in ranked_ids[:top_k]:
        print(f"  {id_to_character[token_id]!r:>6}: {probabilities[token_id]:.6f}")


for starting_text in ["the ", "the c", "the cat ", unusual_prompt]:
    print_top_next_characters(
        starting_text,
        generation_model,
        character_to_id,
        id_to_character,
        top_k=8,
    )
    print()

Starting text: 'the '
Source: observed length 4
Context used: 'the '
     'c': 0.240557
     'd': 0.121272
     'b': 0.101392
     'm': 0.101392
     'r': 0.081511
     's': 0.081511
     'y': 0.081511
     'f': 0.061630

Starting text: 'the c'
Source: observed length 5
Context used: 'the c'
     'a': 0.496503
     'h': 0.356643
    '\n': 0.006993
     ' ': 0.006993
     '.': 0.006993
     'b': 0.006993
     'c': 0.006993
     'd': 0.006993

Starting text: 'the cat '
Source: observed length 8
Context used: 'the cat '
     's': 0.373494
     'a': 0.132530
     'l': 0.132530
     'r': 0.132530
    '\n': 0.012048
     ' ': 0.012048
     '.': 0.012048
     'b': 0.012048

Starting text: 'the mat the'
Source: observed length 6
Context used: 'at the'
     ' ': 0.650794
    '\n': 0.015873
     '.': 0.015873
     'a': 0.015873
     'b': 0.015873
     'c': 0.015873
     'd': 0.015873
     'e': 0.015873



## Compare Sampling and Greedy Decoding

Sampling follows the probability weights, while greedy decoding always selects the largest probability.

Greedy decoding is deterministic but can repeat a high-probability loop.


In [22]:
def generate_greedily(
    starting_text: str,
    number_of_new_tokens: int,
    model: BackoffNGramCharacterModel,
    character_to_id: dict[str, int],
    id_to_character: dict[int, str],
) -> str:
    generated_ids = encode_characters(starting_text, character_to_id)
    for _ in range(number_of_new_tokens):
        probabilities, source, context = model.probabilities_and_source(generated_ids)
        next_id = max(range(len(probabilities)), key=probabilities.__getitem__)
        generated_ids.append(next_id)
    return decode_character_ids(generated_ids, id_to_character)


sampled_text, sampled_sources = generate_text(
    "the ",
    120,
    generation_model,
    character_to_id,
    id_to_character,
    RANDOM_SEED,
)
greedy_text = generate_greedily(
    "the ",
    120,
    generation_model,
    character_to_id,
    id_to_character,
)

print("Sampled generation:")
print(sampled_text)
print()
print("Greedy generation:")
print(greedy_text)

Sampled generation:
the me hayag imaild bcat sleat.
thn.
the hachfo
the catlon the matat s.
the ch mad at tn in 
thhildsat on on n the yard.
tat

Greedy generation:
the cat sat on the mat.
the dog sat on the mat.
the dog sat on the mat.
the dog sat on the mat.
the dog sat on the mat.
the 


Neither decoding method changes the fitted probabilities, and neither should be judged from a single output alone.


## Different Seeds Follow Different Paths

The model probabilities stay fixed while random samples can select different continuations.


In [23]:
for seed in [19, 20, 21]:
    seeded_text, seeded_sources = generate_text(
        "the ",
        80,
        generation_model,
        character_to_id,
        id_to_character,
        seed,
    )
    print(f"seed {seed}: {seeded_text!r}")

seed 19: 'the me hayag imaild bcat sleat.\nthn.\nthe hachfo\nthe catlon the matat s.\nthe ch mad a'
seed 20: 'the trhildngbiry cug.\nthe hd l bd.\nthe dog \nthe dog rbirioug.\ntheyg s\nthehe bifooat '
seed 21: 'the childed am\nthe caaugn e hat iogn the drd.\nuon the rkrug.\ntya biyarde mat.\ntppy b'


## Training and Generation Are Different Phases

Training transforms corpus examples into count dictionaries and probability rows.

Generation holds those fitted rows fixed while choosing tokens from them.

This model does not update its counts from its own generated output.


## Complete Generation Pipeline

The final cell rebuilds the model, generates deterministically, and verifies output vocabulary and source accounting.


In [24]:
final_vocabulary, final_character_to_id, final_id_to_character = (
    build_character_mappings(training_text)
)
final_training_ids = encode_characters(training_text, final_character_to_id)
final_probabilities_by_length = build_probabilities_by_length(
    final_training_ids,
    MAXIMUM_CONTEXT_LENGTH,
    len(final_vocabulary),
    ALPHA,
)
final_unigram = smooth_count_list(
    count_token_ids(final_training_ids, len(final_vocabulary)),
    ALPHA,
)
final_uniform = uniform_probabilities(len(final_vocabulary))
final_model = BackoffNGramCharacterModel(
    MAXIMUM_CONTEXT_LENGTH,
    final_probabilities_by_length,
    final_unigram,
    final_uniform,
    fallback_strategy="unigram",
)
final_text, final_sources = generate_text(
    "the ",
    100,
    final_model,
    final_character_to_id,
    final_id_to_character,
    RANDOM_SEED,
)

assert set(final_text).issubset(set(final_vocabulary))
assert sum(final_sources.values()) == 100
print("Vocabulary size:", len(final_vocabulary))
print("Maximum context length:", MAXIMUM_CONTEXT_LENGTH)
print("Smoothing alpha:", ALPHA)
print("Probability sources:", dict(final_sources))
print("Generated text:", repr(final_text))

Vocabulary size: 23
Maximum context length: 8
Smoothing alpha: 0.1
Probability sources: {'observed length 4': 14, 'observed length 5': 9, 'observed length 1': 21, 'observed length 2': 22, 'observed length 3': 16, 'observed length 6': 7, 'observed length 7': 5, 'observed length 8': 6}
Generated text: 'the me hayag imaild bcat sleat.\nthn.\nthe hachfo\nthe catlon the matat s.\nthe ch mad at tn in \nthhildsat o'


## Common Mistakes

- Append each chosen token before making the next prediction.

- Select context from the end of the current generated token sequence.

- Encode a prompt only if every character belongs to the model vocabulary.

- Use smoothing for unseen targets after observed contexts and backoff for unseen context tuples.

- Expose the probability source when debugging backoff behavior.

- Distinguish sampling from greedy decoding.

- Remember that generation consumes a fitted model and does not retrain it.


## Takeaways

N-gram generation repeatedly maps the rolling context to probabilities, chooses one token, appends it, and slides the context forward.

Shorter-context backoff lets the generator use the longest observed suffix available, while unigram or uniform fallback handles the absence of every positive-length context.

Smoothing and backoff are complementary because they address unseen targets and unseen contexts, respectively.

Sampling provides weighted variety, whereas greedy decoding always takes the most likely token and can become repetitive.

The next chapter returns to evaluation by studying sequence probability, log probability, and the numerical problems caused by multiplying many small probabilities.
